# 02 Recovery Index Pilot

Update the file names and column maps below after completing the inventory notebook. This notebook standardizes monthly dates, builds a master monthly panel, and calculates recovery indices where the 2018-2019 average equals 100.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Configure Raw Files and Column Maps

Replace the example file names and raw column names with the actual manually downloaded files. Leave values blank only until raw data has been collected.

In [ ]:
from src.data_loader import find_raw_file_by_stem

visitor_file = find_raw_file_by_stem(RAW_DIR, "visitor_arrivals")
retail_file = find_raw_file_by_stem(RAW_DIR, "retail_sales")
hotel_file = find_raw_file_by_stem(RAW_DIR, "hotel_performance")

print(f"Visitor file: {visitor_file.name}")
print(f"Retail file: {retail_file.name}")
print(f"Hotel file: {hotel_file.name}")

visitor_column_map = {
    "Month": "month",
    "Total arrivals": "visitor_arrivals",
}

retail_column_map = {
    "Month": "month",
    "Sales value": "retail_sales_value",
    # "Category": "retail_category",
}

hotel_column_map = {
    "Month": "month",
    "Occupancy rate": "hotel_occupancy_rate",
    "Average room rate": "hotel_room_rate",
}

In [ ]:
from src.data_loader import aggregate_monthly, load_monthly_dataset, merge_monthly_panel, save_processed
from src.index_builder import calculate_recovery_indices, select_index_columns
from src.plotting import plot_recovery_indices

visitors = load_monthly_dataset(
    visitor_file,
    visitor_column_map,
    required_columns=["month", "visitor_arrivals"],
)

retail = load_monthly_dataset(
    retail_file,
    retail_column_map,
    required_columns=["month", "retail_sales_value"],
)
retail_monthly = aggregate_monthly(retail, ["retail_sales_value"], agg="sum")

hotel = load_monthly_dataset(
    hotel_file,
    hotel_column_map,
    required_columns=["month"],
    value_columns=["hotel_occupancy_rate", "hotel_room_rate"],
)
hotel_indicator_columns = [
    col for col in ["hotel_occupancy_rate", "hotel_room_rate"] if col in hotel.columns
]
missing_hotel_indicators = sorted(set(["hotel_occupancy_rate", "hotel_room_rate"]) - set(hotel_indicator_columns))
if not hotel_indicator_columns:
    raise ValueError("Hotel data must include occupancy rate, room rate, or both after column mapping.")
if missing_hotel_indicators:
    print(f"Missing hotel indicators: {missing_hotel_indicators}")
else:
    print("Missing hotel indicators: none")

hotel_monthly = aggregate_monthly(hotel, hotel_indicator_columns, agg="mean")

panel = merge_monthly_panel(visitors, retail_monthly, hotel_monthly)
save_processed(panel, PROCESSED_DIR / "master_monthly_panel.csv")
panel.head()

In [ ]:
value_columns = [
    "visitor_arrivals",
    "retail_sales_value",
] + hotel_indicator_columns

indexed = calculate_recovery_indices(panel, value_columns)
indexed.to_csv(TABLE_DIR / "recovery_indices.csv", index=False)
indexed.head()

In [ ]:
index_columns = select_index_columns(indexed)
fig, ax = plot_recovery_indices(
    indexed,
    index_columns,
    output_path=FIGURE_DIR / "recovery_indices.png",
)